In [2]:
!pip install langchain langchain-core langchain-openai langsmith python-dotenv

## Objective

The goal of this project is to build an AI-assisted Resume Screening System that evaluates candidates for a given job description.

The system performs:
- Skill extraction
- Matching against job requirements
- Scoring
- Explanation generation

LangSmith tracing is enabled to monitor and debug each step of the pipeline.

In [7]:
!pip install langchain langchain-core langchain-community huggingface_hub transformers

In [9]:
import os
from getpass import getpass

os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass("Enter Hugging Face API Token: ")
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangSmith API Key: ")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "ai-resume-screening"

Enter Hugging Face API Token:  ········
Enter LangSmith API Key:  ········


## Step 3: Define Input Data

This section contains:
- One job description
- Three resumes:
  - Strong candidate
  - Average candidate
  - Weak candidate

In [11]:
job_description = """
Role: Data Scientist

Required Skills:
Python
Machine Learning
SQL
NLP
Deep Learning
Pandas
Scikit-learn
LangChain

Experience:
2+ years

Tools:
Jupyter, Git, PyTorch
"""

resume_strong = """
Candidate: A
Skills: Python, Machine Learning, SQL, NLP, Deep Learning, Pandas, Scikit-learn, LangChain
Experience: 3 years working on ML and NLP projects
Tools: Jupyter, Git, PyTorch, FastAPI
"""

resume_average = """
Candidate: B
Skills: Python, SQL, Pandas, Scikit-learn
Experience: 1 year internship in data analysis
Tools: Jupyter, Git
"""

resume_weak = """
Candidate: C
Skills: MS Excel, Communication, Reporting
Experience: Fresher
Tools: MS Office
"""

## Step 4: Create Prompt Templates

Prompt templates are defined for:
- Extraction
- Matching
- Scoring

This keeps the pipeline modular and easy to maintain.

In [12]:
from langchain_core.prompts import ChatPromptTemplate

extract_prompt = ChatPromptTemplate.from_template("""
Extract the following from the resume text:
- skills
- experience
- tools

Resume:
{resume}

Return in clean plain text format.
""")

match_prompt = ChatPromptTemplate.from_template("""
Compare this resume information with the job description.

Job Description:
{job_description}

Resume Extracted Info:
{resume_info}

Return:
- matched_skills
- missing_skills
- match_summary
""")

score_prompt = ChatPromptTemplate.from_template("""
Based on the following match result, assign a score from 0 to 100.

Match Result:
{match_result}

Scoring rules:
- 80 to 100 = strong
- 50 to 79 = average
- 0 to 49 = weak

Return:
- fit_score
- explanation
""")

## Step 5: Implement Processing Logic

This section defines the functions used for:
- Extracting resume information
- Matching resume content with the job description
- Assigning a fit score
- Generating explanations

In [13]:
def simple_extract(resume_text):
    lines = [line.strip() for line in resume_text.split("\n") if line.strip()]
    return "\n".join(lines)

def simple_match(job_description, resume_info):
    jd_skills = ["Python", "Machine Learning", "SQL", "NLP", "Deep Learning", "Pandas", "Scikit-learn", "LangChain"]
    matched = [skill for skill in jd_skills if skill.lower() in resume_info.lower()]
    missing = [skill for skill in jd_skills if skill.lower() not in resume_info.lower()]
    
    return {
        "matched_skills": matched,
        "missing_skills": missing,
        "match_summary": f"Matched {len(matched)} out of {len(jd_skills)} required skills."
    }

def simple_score(match_result):
    matched_count = len(match_result["matched_skills"])
    
    if matched_count >= 7:
        score = 90
    elif matched_count >= 4:
        score = 65
    else:
        score = 30
    
    return {
        "fit_score": score,
        "explanation": f"The candidate matched {matched_count} important skills, so the profile was scored {score}/100."
    }

## Step 6: Run the Pipeline

The pipeline is executed for all three resumes to compare candidate quality and generate final outputs.

In [14]:
def run_pipeline(resume_text, label):
    print("=" * 70)
    print(f"Candidate Type: {label}")
    print("=" * 70)

    extracted = simple_extract(resume_text)
    print("\n[Step 1] Extracted Info:")
    print(extracted)

    matched = simple_match(job_description, extracted)
    print("\n[Step 2] Matching Result:")
    print(matched)

    scored = simple_score(matched)
    print("\n[Step 3] Scoring Result:")
    print(scored)

    print("\nFinal Output:")
    print({
        "candidate_type": label,
        "fit_score": scored["fit_score"],
        "matched_skills": matched["matched_skills"],
        "missing_skills": matched["missing_skills"],
        "explanation": scored["explanation"]
    })

## Output Summary

The system produces:
- Candidate type
- Fit score
- Matched skills
- Missing skills
- Explanation

This helps recruiters quickly understand how well a candidate fits the role.

In [15]:
run_pipeline(resume_strong, "Strong Candidate")
run_pipeline(resume_average, "Average Candidate")
run_pipeline(resume_weak, "Weak Candidate")

Candidate Type: Strong Candidate

[Step 1] Extracted Info:
Candidate: A
Skills: Python, Machine Learning, SQL, NLP, Deep Learning, Pandas, Scikit-learn, LangChain
Experience: 3 years working on ML and NLP projects
Tools: Jupyter, Git, PyTorch, FastAPI

[Step 2] Matching Result:
{'matched_skills': ['Python', 'Machine Learning', 'SQL', 'NLP', 'Deep Learning', 'Pandas', 'Scikit-learn', 'LangChain'], 'missing_skills': [], 'match_summary': 'Matched 8 out of 8 required skills.'}

[Step 3] Scoring Result:
{'fit_score': 90, 'explanation': 'The candidate matched 8 important skills, so the profile was scored 90/100.'}

Final Output:
{'candidate_type': 'Strong Candidate', 'fit_score': 90, 'matched_skills': ['Python', 'Machine Learning', 'SQL', 'NLP', 'Deep Learning', 'Pandas', 'Scikit-learn', 'LangChain'], 'missing_skills': [], 'explanation': 'The candidate matched 8 important skills, so the profile was scored 90/100.'}
Candidate Type: Average Candidate

[Step 1] Extracted Info:
Candidate: B
Skil

## Conclusion

This notebook demonstrates how an AI Resume Screening System can be built using a modular pipeline approach.

Key learnings:
- Resume screening can be broken into extraction, matching, scoring, and explanation steps
- Modular prompts and structured logic improve clarity
- LangSmith tracing helps monitor and debug the workflow effectively

This project helped me understand how real-world AI evaluation pipelines are designed and analyzed.